Imports

In [0]:
from pyspark.sql.functions import *
from delta.tables import *

In [0]:
%sql
/* ALTER TABLE retailanalytics.config.table_config
ADD COLUMNS (
    BusinessKey STRING
);*/

In [0]:
%sql
UPDATE retailanalytics.config.table_config
SET BusinessKey = 'CustomerID'
WHERE TableName = 'customers';

UPDATE retailanalytics.config.table_config
SET BusinessKey = 'ProductID'
WHERE TableName = 'products';

UPDATE retailanalytics.config.table_config
SET BusinessKey = 'OrderID'
WHERE TableName = 'orders';

num_affected_rows
1


In [0]:
%sql
SELECT * FROM retailanalytics.config.table_config;

TableName,SourceFile,SourceFormat,LoadType,SCDType,ActiveFlag,BusinessKey
orders,orders.csv,csv,INCREMENTAL,NA,Y,OrderID
products,products.csv,csv,FULL,NA,Y,ProductID
exchange_rates,exchange_rates.json,json,FULL,NA,Y,null
customers,customers.csv,csv,INCREMENTAL,SCD2,Y,CustomerID


In [0]:
%sql
UPDATE retailanalytics.config.table_config
SET SCDType = 'SCD2'
WHERE TableName = 'products';

UPDATE retailanalytics.config.table_config
SET SCDType = 'NA'
WHERE TableName = 'exchange_rates';

num_affected_rows
1


Read Metadata

In [0]:
config_df = spark.table("retailanalytics.config.table_config")

scd_tables = (
    config_df.filter(col("SCDType") == "SCD2").filter(col("ActiveFlag") == "Y")
    .select("TableName","BusinessKey").collect())

print(scd_tables)

[Row(TableName='products', BusinessKey='ProductID'), Row(TableName='customers', BusinessKey='CustomerID')]


Dynamic SCD2 Merge

In [0]:
for row in scd_tables:

    table_name = row["TableName"]
    business_key = row["BusinessKey"]
    source_df = spark.table(f"retailanalytics.silver.{table_name}")
    history_table = (f"retailanalytics.history.dim_{table_name}_hist")
    history_df = spark.table(history_table)
    active_df = (history_df.filter(col("IsCurrent") == True))

    exclude_cols = [business_key,"CustomerSK","ProductSK","EffectiveDate","ExpiryDate","IsCurrent","_ProcessedTimestamp","_IsRejected","_AdfPipelineRunId","LastUpdated"]

    compare_cols = [
        c
        for c in source_df.columns
        if c in history_df.columns
        and c not in exclude_cols
    ]

    source_hash = sha2(
        concat_ws("||",*[coalesce(col(f"s.{c}").cast("string"),lit("")) for c in compare_cols]),256)

    history_hash = sha2(
        concat_ws("||",*[coalesce(col(f"h.{c}").cast("string"),lit("")) for c in compare_cols]),256)

    changed_df = (source_df.alias("s").join(active_df.alias("h"),business_key,"inner")
        .filter(source_hash != history_hash).select("s.*"))

    # Freeze dataframe before merge
    changed_df = spark.createDataFrame(changed_df.collect(),changed_df.schema)
    change_count = changed_df.count()
    print(f"{table_name} Changed Records :",change_count)

    if change_count > 0:
        DeltaTable.forName(spark,history_table).alias("t").merge(changed_df.alias("s"),
            f"""
            t.{business_key}=s.{business_key}
            AND t.IsCurrent = true """
        ).whenMatchedUpdate(set={"IsCurrent": "false","ExpiryDate": "current_timestamp()"}).execute()

        technical_cols = ["_AdfPipelineRunId","_ProcessedTimestamp", "_IsRejected", "LastUpdated"]
        drop_cols = [c for c in technical_cols if c in changed_df.columns]
        changed_df = changed_df.drop(*drop_cols)
        new_records = (changed_df.withColumn("EffectiveDate",current_timestamp())
        .withColumn("ExpiryDate",lit(None).cast("timestamp"))
        .withColumn("IsCurrent",lit(True)=))

        insert_cols = [
            c
            for c in history_df.columns
            if c not in ["CustomerSK","ProductSK"]]
        new_records = new_records.select(*insert_cols)

        print("======== DEBUG ========")
        print("Table :", table_name)
        print("Changed Count :", change_count)
        print("New Record Count :")
        print(new_records.count())
        new_records.printSchema()
        display(new_records)
        (new_records.write.format("delta").mode("append").saveAsTable(history_table))

        print(f"SCD2 Merge Completed : {table_name}")

products Changed Records : 0
customers Changed Records : 1
======== DEBUG ========
Table : customers
Changed Count : 1
New Record Count :
1
root
 |-- CustomerID: integer (nullable = true)
 |-- FirstName: string (nullable = true)
 |-- LastName: string (nullable = true)
 |-- Email: string (nullable = true)
 |-- Phone: string (nullable = true)
 |-- City: string (nullable = true)
 |-- State: string (nullable = true)
 |-- EffectiveDate: timestamp (nullable = false)
 |-- ExpiryDate: timestamp (nullable = true)
 |-- IsCurrent: boolean (nullable = false)



CustomerID,FirstName,LastName,Email,Phone,City,State,EffectiveDate,ExpiryDate,IsCurrent
9,Karthik,Patel,customer9@gmail.com,9696355850,Bangalore,Maharashtra,2026-06-25T05:56:16.948Z,null,true


SCD2 Merge Completed : customers


In [0]:
display(spark.table("retailanalytics.history.dim_customers_hist"))
display(spark.table("retailanalytics.history.dim_products_hist"))

CustomerSK,CustomerID,FirstName,LastName,Email,Phone,City,State,EffectiveDate,ExpiryDate,IsCurrent
1,1,Arjun,Patel,customer1@gmail.com,9927622808,Mumbai,Maharashtra,2026-06-25T05:53:35.319Z,null,true
2,2,Arjun,Iyer,customer2@gmail.com,9885934963,Mumbai,Maharashtra,2026-06-25T05:53:35.319Z,null,true
3,3,Anjali,Patel,customer3@gmail.com,9959015427,Delhi,Delhi,2026-06-25T05:53:35.319Z,null,true
4,4,Rahul,Singh,customer4@gmail.com,9186064385,Chennai,Tamil Nadu,2026-06-25T05:53:35.319Z,null,true
5,5,Divya,Reddy,customer5@gmail.com,9807398273,Chennai,Tamil Nadu,2026-06-25T05:53:35.319Z,null,true
6,6,Karthik,Reddy,customer6@gmail.com,9432222084,Bangalore,Karnataka,2026-06-25T05:53:35.319Z,null,true
7,7,Karthik,Sharma,customer7@gmail.com,9302502246,Bangalore,Karnataka,2026-06-25T05:53:35.319Z,null,true
8,8,Anjali,Reddy,customer8@gmail.com,9996434071,Mumbai,Maharashtra,2026-06-25T05:53:35.319Z,null,true
9,9,Karthik,Patel,customer9@gmail.com,9696355850,Mumbai,Maharashtra,2026-06-25T05:53:35.319Z,null,true
10,10,Priya,Patel,customer10@gmail.com,9796367641,Bangalore,Karnataka,2026-06-25T05:53:35.319Z,null,true


ProductSK,ProductID,ProductName,Category,SubCategory,Brand,CostPrice,EffectiveDate,ExpiryDate,IsCurrent
1,1,Laptop_1,Electronics,Laptop,Sony,252.44,2026-06-25T05:53:38.882Z,null,true
2,2,Jacket_2,Fashion,Jacket,Samsung,679.93,2026-06-25T05:53:38.882Z,null,true
3,3,Tablet_3,Electronics,Tablet,Nike,41.46,2026-06-25T05:53:38.882Z,null,true
4,4,Laptop_4,Electronics,Laptop,Apple,510.30,2026-06-25T05:53:38.882Z,null,true
5,5,Tablet_5,Electronics,Tablet,Apple,718.86,2026-06-25T05:53:38.882Z,null,true
6,6,Football_6,Sports,Football,Nike,593.37,2026-06-25T05:53:38.882Z,null,true
7,7,Laptop_7,Electronics,Laptop,Nike,346.85,2026-06-25T05:53:38.882Z,null,true
8,8,Shirt_8,Fashion,Shirt,Sony,111.19,2026-06-25T05:53:38.882Z,null,true
9,9,Football_9,Sports,Football,Sony,849.02,2026-06-25T05:53:38.882Z,null,true
10,10,Chair_10,Home,Chair,Nike,540.87,2026-06-25T05:53:38.882Z,null,true


In [0]:
%sql
UPDATE retailanalytics.silver.customers
SET City = 'Bangalore'
WHERE CustomerID = 9;

num_affected_rows
1


In [0]:
display(spark.table("retailanalytics.history.dim_customers_hist").filter(col("CustomerID") == 9))

CustomerSK,CustomerID,FirstName,LastName,Email,Phone,City,State,EffectiveDate,ExpiryDate,IsCurrent
9,9,Karthik,Patel,customer9@gmail.com,9696355850,Mumbai,Maharashtra,2026-06-25T05:53:35.319Z,2026-06-25T05:56:11.207Z,false
1002,9,Karthik,Patel,customer9@gmail.com,9696355850,Bangalore,Maharashtra,2026-06-25T05:56:17.610Z,null,true


# Manual Approach

In [0]:
"""from pyspark.sql.functions import *

new_customer = (

    spark.table("retailanalytics.silver.customers")

    .filter(col("CustomerID") == 3)

    .select(
        "CustomerID",
        "FirstName",
        "LastName",
        "Email",
        "Phone",
        "City",
        "State"
    )

    .withColumn(
        "EffectiveDate",
        current_timestamp()
    )

    .withColumn(
        "ExpiryDate",
        lit(None).cast("timestamp")
    )

    .withColumn(
        "IsCurrent",
        lit(True)
    )

)

(new_customer.write
 .format("delta")
 .mode("append")
 .saveAsTable(
     "retailanalytics.history.dim_customers_hist"
 ))""""